# Inspector
Provides info on the run-time environment.

In [1]:
import os
import json
import requests
from pathlib import Path

# 1. Fetch current live PyCharm-Colab session details
jupyter_ip = os.environ.get('COLAB_JUPYTER_IP', '172.28.0.12')
session_data = requests.get(f"http://{jupyter_ip}:9000/api/sessions").json()[0]

session_id = session_data['id']
notebook_name = session_data['name']

# 2. Path to the colab-cli local registry
cli_config_path = Path.home() / ".config" / "colab-cli" / "sessions.json"
cli_config_path.parent.mkdir(parents=True, exist_ok=True)

# 3. Load existing or initialize empty sessions structure
if cli_config_path.exists() and cli_config_path.stat().st_size > 0:
    with open(cli_config_path, "r") as f:
        try:
            sessions_db = json.load(f)
        except json.JSONDecodeError:
            sessions_db = {}
else:
    sessions_db = {}

# 4. Inject PyCharm's backend tracking data into the schema
# (Adjust keys if your specific version uses a list instead of a dictionary mapping)
sessions_db[notebook_name] = {
    "session_id": session_id,
    "backend_ip": jupyter_ip,
    "hardware": "GPU/CPU (PyCharm Local Link)",
    "status": "active"
}

with open(cli_config_path, "w") as f:
    json.dump(sessions_db, f, indent=4)

print(f"Successfully registered '{notebook_name}' to ~/.config/colab-cli/sessions.json")


Successfully registered 'experiment-intellij-f49bde58-f91f-4584-abcb-2c5eb6f77bca.ipynb' to ~/.config/colab-cli/sessions.json
